In [ ]:
# Clone or pull repo, install deps, set device (GPU if available)
import sys
from pathlib import Path

REPO_URL = "https://github.com/PulockDas/Secure-Inference-Token-Reduced-VIT.git"  # set your repo
PROJECT_DIR = "/content/Secure-Inference-Token-Reduced-VIT"

if Path(PROJECT_DIR).exists():
    %cd $PROJECT_DIR
    !git pull
else:
    !git clone $REPO_URL $PROJECT_DIR
    %cd $PROJECT_DIR
!pip install -q -r requirements.txt

sys.path.insert(0, PROJECT_DIR)

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
from data import get_lc25000_root, get_dataloaders

root = get_lc25000_root()  # public dataset, no auth needed
train_loader, val_loader, test_loader = get_dataloaders(
    root_dir=root, batch_size=32, val_ratio=0.15, test_ratio=0.15, seed=42,
    subdir_depth=2, image_size=224, num_workers=2,
)
ds = train_loader.dataset
print("Classes:", ds.class_names)
print("Train:", len(ds), "Val:", len(val_loader.dataset), "Test:", len(test_loader.dataset))

In [ ]:
from models import get_teacher_vit
from training import train_teacher

num_classes = train_loader.dataset.num_classes
model = get_teacher_vit(num_classes=num_classes, pretrained=True)
model = train_teacher(
    model, train_loader, val_loader, device,
    epochs=10, lr=1e-4,
    log_path=f"{PROJECT_DIR}/logs/teacher.csv",
    checkpoint_dir=f"{PROJECT_DIR}/checkpoints",
)

In [ ]:
# Load best teacher from checkpoint and evaluate on test set
from training import load_teacher_checkpoint, evaluate_teacher

ckpt_path = f"{PROJECT_DIR}/checkpoints/teacher_best.pt"
model = load_teacher_checkpoint(ckpt_path, device)
result = evaluate_teacher(
    model, test_loader, device, train_loader.dataset.class_names,
    results_dir=f"{PROJECT_DIR}/results"
)
print("Test accuracy:", f"{result['test_acc']:.4f}")
print("Per-class accuracy:")
for name, acc in zip(result["class_names"], result["per_class_acc"]):
    print(f"  {name}: {acc:.4f}")

In [ ]:
# Must change the token
GITHUB_TOKEN = "your_token_here"
!git config --global user.name "PulockDas"
!git config --global user.email "pulockkamol50@gmail.com"

In [ ]:
# Push results, checkpoints, and logs to GitHub
%cd $PROJECT_DIR
!git add results/ checkpoints/ logs/
!git commit -m "Add teacher training results, checkpoints, and evaluation" || echo "No changes to commit"
# uncomment below and add your token:
!git push https://{GITHUB_TOKEN}@github.com/PulockDas/Secure-Inference-Token-Reduced-VIT.git